<a href="https://colab.research.google.com/github/ekee7/acc_reality_check/blob/main/ACC_Reality_Check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **ACC Reality Check**

**Group Members: Emma Kee (hvh6ng), Kylie Sharp (dhq4mr), and Raffaella Alpaca (bet3at)**

In [1]:
from pathlib import Path

# Create folders for raw data
Path("data/raw/torvik").mkdir(parents=True, exist_ok=True)
Path("data/raw/scoreboard").mkdir(parents=True, exist_ok=True)
Path("data/raw/boxscores").mkdir(parents=True, exist_ok=True)

print("Project folders created")

Project folders created


## **Import Data**

In [2]:
import requests
import pandas as pd

# Download Torvik data
url = "https://raw.githubusercontent.com/ekee7/acc_reality_check/main/2027_team_results.csv"

response = requests.get(url)
response.raise_for_status()

# Save original CSV
file_path = Path("data/raw/torvik/2027_team_results.csv")
file_path.write_bytes(response.content)

# Load CSV into pandas
df_torvik = pd.read_csv(file_path)

# Preserve original copy
df_torvik_original = df_torvik.copy()

print("Dataset downloaded successfully")
print("Dataset shape:", df_torvik.shape)

display(df_torvik.head())

Dataset downloaded successfully
Dataset shape: (365, 45)


,rank,team,conf,record,adjoe,oe Rank,adjde,de Rank,barthag,rank.1,...,ConPA,ConPoss,ConOE,ConDE,ConSOSRemain,Conf Win%,WAB,WAB Rk,Fun Rk,adjt
0,1,Houston,B12,6-0,124.474683,3,93.804798,3,0.962788,1,...,0.0,0.0,0,0,0.831863,0,2.444834,1,26,67.558509
1,2,Illinois,B10,5-1,126.281293,1,96.602043,16,0.956102,2,...,0.0,0.0,0,0,0.836524,0,0.985492,20,120,73.530920
2,3,Kansas,B12,5-1,122.691516,6,94.294347,4,0.953797,3,...,0.0,0.0,0,0,0.829842,0,1.296530,9,161,72.710877
3,4,Florida,SEC,6-1,124.762015,2,96.285008,13,0.951644,4,...,0.0,0.0,0,0,0.856907,0,2.286053,2,68,68.495192
4,5,Duke,ACC,4-2,119.887207,20,93.240421,1,0.947388,5,...,0.0,0.0,0,0,0.801220,0,-0.309751,88,336,71.564028


In [3]:
# Overview of df_torvik
print("Number of rows:", len(df_torvik))
print("Number of columns:", len(df_torvik.columns))

print("\nColumn names:")
print(df_torvik.columns.tolist())

Number of rows: 365
Number of columns: 45

Column names:
['rank', 'team', 'conf', 'record', 'adjoe', 'oe Rank', 'adjde', 'de Rank', 'barthag', 'rank.1', 'proj. W', 'Proj. L', 'Pro Con W', 'Pro Con L', 'Con Rec.', 'sos', 'ncsos', 'consos', 'Proj. SOS', 'Proj. Noncon SOS', 'Proj. Con SOS', 'elite SOS', 'elite noncon SOS', 'Opp OE', 'Opp DE', 'Opp Proj. OE', 'Opp Proj DE', 'Con Adj OE', 'Con Adj DE', 'Qual O', 'Qual D', 'Qual Barthag', 'Qual Games', 'FUN', 'ConPF', 'ConPA', 'ConPoss', 'ConOE', 'ConDE', 'ConSOSRemain', 'Conf Win%', 'WAB', 'WAB Rk', 'Fun Rk', 'adjt']


In [4]:
# Filter the dataset to ACC teams
df_acc = df_torvik[df_torvik["conf"] == "ACC"].copy()

# Check the results
print("Number of ACC teams:", len(df_acc))
print("\nACC teams:")
display(df_acc[["team", "conf", "record"]].sort_values("team"))

Number of ACC teams: 18

ACC teams:


,team,conf,record
100,Boston College,ACC,6-0
99,California,ACC,4-2
67,Clemson,ACC,4-1
4,Duke,ACC,4-2
54,Florida St.,ACC,5-1
122,Georgia Tech,ACC,5-1
23,Louisville,ACC,3-1
14,Miami FL,ACC,5-1
32,N.C. State,ACC,5-1
51,North Carolina,ACC,4-1


In [5]:
# Save the ACC teams to a separate CSV file
acc_file_path = Path("data/raw/torvik/acc_teams.csv")
df_acc.to_csv(acc_file_path, index=False)

print("ACC dataset saved successfully")
print("Number of teams saved:", len(df_acc))

ACC dataset saved successfully
Number of teams saved: 18


In [6]:
from datetime import date, timedelta
import json

# ESPN scoreboard API
scoreboard_url = ("https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard")

# Set the date range
start_date = date(2026, 11, 1)
end_date = date(2026, 11, 23)

# Store all scoreboard responses
all_scoreboards = {}

current_date = start_date

while current_date <= end_date:
    date_string = current_date.strftime("%Y%m%d")

    params = {
        "groups": 2,
        "dates": date_string,
        "limit": 200
    }

    response = requests.get(scoreboard_url, params=params)
    response.raise_for_status()

    scoreboard_data = response.json()

    # Save the original ESPN response
    file_path = Path(f"data/raw/scoreboard/scoreboard_{date_string}.json")

    file_path.write_text(json.dumps(scoreboard_data, indent=2))

    all_scoreboards[date_string] = scoreboard_data

    print(
        date_string,
        "- Games found:",
        len(scoreboard_data.get("events", []))
    )

    current_date += timedelta(days=1)

print("\nScoreboard data collection complete")

20261101 - Games found: 1
20261102 - Games found: 13
20261103 - Games found: 4
20261104 - Games found: 1
20261105 - Games found: 7
20261106 - Games found: 9
20261107 - Games found: 1
20261108 - Games found: 3
20261109 - Games found: 5
20261110 - Games found: 10
20261111 - Games found: 2
20261112 - Games found: 0
20261113 - Games found: 11
20261114 - Games found: 3
20261115 - Games found: 3
20261116 - Games found: 2
20261117 - Games found: 10
20261118 - Games found: 3
20261119 - Games found: 5
20261120 - Games found: 9
20261121 - Games found: 3
20261122 - Games found: 2
20261123 - Games found: 3

Scoreboard data collection complete


In [7]:
# Combine games from all saved scoreboard responses
all_games = []

for date_string, scoreboard in all_scoreboards.items():
    for game in scoreboard.get("events", []):
        all_games.append(game)

# Create a DataFrame with basic game information
games_summary = []

for game in all_games:
    competition = game["competitions"][0]

    teams = competition["competitors"]

    games_summary.append({
        "game_id": game["id"],
        "game_name": game["name"],
        "game_date_utc": game["date"],
        "status": competition["status"]["type"]["name"],
        "home_team": next(
            (team["team"]["displayName"] for team in teams
             if team["homeAway"] == "home"), None
        ),
        "away_team": next(
            (team["team"]["displayName"] for team in teams
             if team["homeAway"] == "away"), None
        )
    })

df_games_summary = pd.DataFrame(games_summary)

print("Total games collected:", len(df_games_summary))
print("\nGame statuses:")
print(df_games_summary["status"].value_counts())

display(df_games_summary.head(10))

Total games collected: 110

Game statuses:
status
STATUS_FINAL    110
Name: count, dtype: int64


,game_id,game_name,game_date_utc,status,home_team,away_team
0,401920982,Notre Dame Fighting Irish at Villanova Wildcats,2026-11-01T14:30Z,STATUS_FINAL,Villanova Wildcats,Notre Dame Fighting Irish
1,401906769,Miami Hurricanes at Florida Gators,2026-11-02T23:00Z,STATUS_FINAL,Florida Gators,Miami Hurricanes
2,401909562,Army Black Knights at Duke Blue Devils,2026-11-02T23:30Z,STATUS_FINAL,Duke Blue Devils,Army Black Knights
3,401918947,Chattanooga Mocs at Louisville Cardinals,2026-11-03T02:00Z,STATUS_FINAL,Louisville Cardinals,Chattanooga Mocs
4,401918073,Stetson Hatters at Clemson Tigers,2026-11-03T00:00Z,STATUS_FINAL,Clemson Tigers,Stetson Hatters
5,401913099,Western Carolina Catamounts at North Carolina ...,2026-11-03T00:00Z,STATUS_FINAL,North Carolina Tar Heels,Western Carolina Catamounts
6,401909716,New Haven Chargers at Syracuse Orange,2026-11-03T00:00Z,STATUS_FINAL,Syracuse Orange,New Haven Chargers
7,401909705,Rider Broncs at Pittsburgh Panthers,2026-11-03T00:00Z,STATUS_FINAL,Pittsburgh Panthers,Rider Broncs
8,401918955,Campbell Fighting Camels at NC State Wolfpack,2026-11-03T00:30Z,STATUS_FINAL,NC State Wolfpack,Campbell Fighting Camels
9,401918967,Southern Jaguars at SMU Mustangs,2026-11-03T01:00Z,STATUS_FINAL,SMU Mustangs,Southern Jaguars


In [8]:
# Keep only completed games
df_games_final = df_games_summary[
    df_games_summary["status"] == "STATUS_FINAL"
].copy()

# Check the results
print("Total games collected:", len(df_games_summary))
print("Completed games:", len(df_games_final))
print("Scheduled games excluded:", len(df_games_summary) - len(df_games_final))

display(df_games_final.head())

Total games collected: 110
Completed games: 110
Scheduled games excluded: 0


,game_id,game_name,game_date_utc,status,home_team,away_team
0,401920982,Notre Dame Fighting Irish at Villanova Wildcats,2026-11-01T14:30Z,STATUS_FINAL,Villanova Wildcats,Notre Dame Fighting Irish
1,401906769,Miami Hurricanes at Florida Gators,2026-11-02T23:00Z,STATUS_FINAL,Florida Gators,Miami Hurricanes
2,401909562,Army Black Knights at Duke Blue Devils,2026-11-02T23:30Z,STATUS_FINAL,Duke Blue Devils,Army Black Knights
3,401918947,Chattanooga Mocs at Louisville Cardinals,2026-11-03T02:00Z,STATUS_FINAL,Louisville Cardinals,Chattanooga Mocs
4,401918073,Stetson Hatters at Clemson Tigers,2026-11-03T00:00Z,STATUS_FINAL,Clemson Tigers,Stetson Hatters


In [9]:
# Create the dated Torvik snapshot

# Add the import statement
from datetime import datetime

# Get the current date
snapshot_date = datetime.now().strftime("%Y-%m-%d")

# Create the filename
snapshot_file = f"data/raw/torvik/torvik_{snapshot_date}.csv"

# Save the full dataset - csv. file
df_torvik.to_csv(snapshot_file, index=False)

# Print summary information
print("Snapshot saved:", snapshot_file)
print("Number of teams:", len(df_torvik))
print("Number of columns:", len(df_torvik.columns))

Snapshot saved: data/raw/torvik/torvik_2026-11-28.csv
Number of teams: 365
Number of columns: 45


In [10]:
# ESPN Box Score Data Collection

# Create import statement
import requests

# Select one of the games to inspect the ESPN box score response
game_id = df_games_final.loc[0, "game_id"]

# Establish the URL - changes based on game_id
url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/summary?event={game_id}"

# Create response variable
response = requests.get(url)

# Check to see the status code based on the response
print("Status Code:", response.status_code)

# Assign the response to the boxscore_data variable
boxscore_data = response.json()

# Print the avail. section information based on the boxscore_data keys
print("Available sections:", boxscore_data.keys())


Status Code: 200
Available sections: dict_keys(['boxscore', 'format', 'gameInfo', 'leaders', 'broadcasts', 'pickcenter', 'odds', 'againstTheSpread', 'news', 'winprobability', 'plays', 'header', 'article', 'videos', 'wallclockAvailable', 'meta', 'standings'])


In [11]:
# ESPN box score collection - download and save the games in JSON format

# Import libraries
from pathlib import Path
import time
import json
import requests

# Create a folder to hold the raw ESPN box scores
boxscore_folder = Path("/content/data/raw/espn/boxscores")
boxscore_folder.mkdir(parents=True, exist_ok=True)

# Keep track of the successful and unsuccessful downloads from ESPN
successful_downloads = 0  # Initialize at 0
failed_downloads = []

# Download the box score for each completed game
for game_id in df_games_final["game_id"]:

    # Create the URL using the unique game ID
    url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/summary?event={game_id}"

    try:
        # Send a request to ESPN and check for a successful response
        response = requests.get(url, timeout=30)
        response.raise_for_status()

        # Convert the response into a Python dictionary
        boxscore_data = response.json()

        # Save the raw JSON response using the game ID as the file name
        file_path = boxscore_folder / f"{game_id}.json"

        with open(file_path, "w", encoding="utf-8") as file:
            json.dump(boxscore_data, file, indent=4)

        # Verify that the file exists and is not empty
        if file_path.exists() and file_path.stat().st_size > 0:
            successful_downloads += 1
        else:
            failed_downloads.append((game_id, "File was not saved"))

    except Exception as error:
        # Record the game ID and error if the download fails
        failed_downloads.append((game_id, str(error)))

    # Use a short pause between requests
    time.sleep(0.2)

# Print summary information
print("Successful downloads:", successful_downloads)
print("Failed downloads:", len(failed_downloads))
print("Files saved:", len(list(boxscore_folder.glob("*.json"))))
print("Files saved in:", boxscore_folder)

Successful downloads: 110
Failed downloads: 0
Files saved: 110
Files saved in: /content/data/raw/espn/boxscores


In [12]:
# Verify the downloaded JSON files

# Get a list of all the box score files
boxscore_files = list(boxscore_folder.glob("*.json"))

# Open one of the saved box scores - using read mode
with open(boxscore_files[0], "r", encoding="utf-8") as file:
    sample_boxscore = json.load(file)

# Display the main sections of the JSON response
print("Available sections:", sample_boxscore.keys())

# Inspect the structure of the box score section
print("\nBox score sections:")
print(sample_boxscore["boxscore"].keys())


Available sections: dict_keys(['boxscore', 'format', 'gameInfo', 'leaders', 'broadcasts', 'pickcenter', 'odds', 'againstTheSpread', 'news', 'header', 'plays', 'winprobability', 'article', 'videos', 'wallclockAvailable', 'meta', 'standings'])

Box score sections:
dict_keys(['teams', 'players'])


In [13]:
# Inspect the team information from the previous sample box score

# Print the number of teams that were included in the box score
print("Number of teams:", len(sample_boxscore["boxscore"]["teams"]))

# Print the info avail for the first team
print("\nFirst team information:")
print(sample_boxscore["boxscore"]["teams"][0].keys())

# Print the actual team data
print("\nTeam data:")
print(sample_boxscore["boxscore"]["teams"][0])

Number of teams: 2

First team information:
dict_keys(['team', 'statistics', 'displayOrder', 'homeAway'])

Team data:
{'team': {'id': '221', 'uid': 's:40~l:41~t:221', 'slug': 'pittsburgh-panthers', 'location': 'Pittsburgh', 'name': 'Panthers', 'abbreviation': 'PITT', 'displayName': 'Pittsburgh Panthers', 'shortDisplayName': 'Pitt', 'color': '003594', 'alternateColor': 'ffb81c', 'logo': 'https://a.espncdn.com/i/teamlogos/ncaa/500/221.png'}, 'statistics': [{'name': 'fieldGoalsMade-fieldGoalsAttempted', 'displayValue': '26-56', 'label': 'FG'}, {'name': 'fieldGoalPct', 'displayValue': '46', 'abbreviation': 'FG%', 'label': 'Field Goal %'}, {'name': 'threePointFieldGoalsMade-threePointFieldGoalsAttempted', 'displayValue': '8-28', 'label': '3PT'}, {'name': 'threePointFieldGoalPct', 'displayValue': '29', 'abbreviation': '3P%', 'label': 'Three Point %'}, {'name': 'freeThrowsMade-freeThrowsAttempted', 'displayValue': '16-20', 'label': 'FT'}, {'name': 'freeThrowPct', 'displayValue': '80', 'abbrev

In [14]:
# Inspect the ESPN players section from the sample box score information

# Print the number of teams included in the player section
print("Number of players:", len(sample_boxscore["boxscore"]["players"]))

# Print the available information for the first team
print("\nFirst team information:")
print(sample_boxscore["boxscore"]["players"][0].keys())

# Print the player information
print("\nPlayer information:")
print(sample_boxscore["boxscore"]["players"][0])

Number of players: 2

First team information:
dict_keys(['team', 'statistics', 'displayOrder'])

Player information:
{'team': {'id': '221', 'uid': 's:40~l:41~t:221', 'slug': 'pittsburgh-panthers', 'location': 'Pittsburgh', 'name': 'Panthers', 'abbreviation': 'PITT', 'displayName': 'Pittsburgh Panthers', 'shortDisplayName': 'Pitt', 'color': '003594', 'alternateColor': 'ffb81c', 'logo': 'https://a.espncdn.com/i/teamlogos/ncaa/500/221.png'}, 'statistics': [{'names': ['MIN', 'PTS', 'FG', '3PT', 'FT', 'REB', 'AST', 'TO', 'STL', 'BLK', 'OREB', 'DREB', 'PF'], 'keys': ['minutes', 'points', 'fieldGoalsMade-fieldGoalsAttempted', 'threePointFieldGoalsMade-threePointFieldGoalsAttempted', 'freeThrowsMade-freeThrowsAttempted', 'rebounds', 'assists', 'turnovers', 'steals', 'blocks', 'offensiveRebounds', 'defensiveRebounds', 'fouls'], 'labels': ['MIN', 'PTS', 'FG', '3PT', 'FT', 'REB', 'AST', 'TO', 'STL', 'BLK', 'OREB', 'DREB', 'PF'], 'descriptions': ['Minutes', 'Points', 'Field Goals Made/Attempted', 

In [15]:
# Inspect the individual player

# Fetch the first team in the player section with the 0 index
first_team_players = sample_boxscore["boxscore"]["players"][0]

# Fetch the first player on that team
first_player = first_team_players["statistics"][0]["athletes"][0]

# Print the player's name
print("Player name:", first_player["athlete"]["displayName"])

# Print the player's ID
print("Player ID:", first_player["athlete"]["id"])

# Print if the player is a starter
print("Starter:", first_player["starter"])

# Print the player's statistics
print("Player statistics:", first_player["stats"])

# Print the labels that are associated with the in-game statistics
print("Statistic labels:", first_team_players["statistics"][0]["labels"])

Player name: Akir Souare
Player ID: 5174559
Starter: True
Player statistics: ['12', '4', '1-2', '0-0', '2-2', '3', '0', '0', '2', '1', '3', '0', '2']
Statistic labels: ['MIN', 'PTS', 'FG', '3PT', 'FT', 'REB', 'AST', 'TO', 'STL', 'BLK', 'OREB', 'DREB', 'PF']


**Data Transformation**

Team Name Crosswalk

The ESPN and Bart Torvik datasets use different naming
conventions for some teams. We will create a crosswalk
to match teams across both data sources and validate
that all 18 ACC schools are correctly matched.

In [16]:
#collect unique ESPN teams
espn_teams = {}

for game in all_games:
    competition = game["competitions"][0]

    for competitor in competition["competitors"]:
        team = competitor["team"]

        espn_teams[team["id"]] = team["location"]

#convert to a DataFrame
df_espn_teams = pd.DataFrame(
    espn_teams.items(),
    columns=["espn_id", "espn_name"]
)

print("Number of unique ESPN teams:", len(df_espn_teams))

display(df_espn_teams.head(20))

Number of unique ESPN teams: 116


,espn_id,espn_name
0,222,Villanova
1,87,Notre Dame
2,57,Florida
3,2390,Miami
4,150,Duke
5,349,Army
6,97,Louisville
7,236,Chattanooga
8,228,Clemson
9,56,Stetson


In [17]:
#normalize team names for automatic matching
def normalize_name(name):
    name = name.lower().strip()
    name = name.replace(".", "")
    name = name.replace("-", " ")

    words = name.split()
    words = ["state" if word == "st" else word for word in words]

    return " ".join(words)
#create copies to preserve the original datasets
espn_matching = df_espn_teams.copy()

torvik_matching = df_torvik[["team", "conf"]].copy()

#normalize names in both datasets
espn_matching["normalized_name"] = espn_matching["espn_name"].apply(normalize_name)

torvik_matching["normalized_name"] = torvik_matching["team"].apply(normalize_name)

#match ESPN teams with Torvik teams
df_crosswalk = espn_matching.merge(
    torvik_matching,
    on="normalized_name",
    how="left"
)
#rename the Torvik team column
df_crosswalk = df_crosswalk.rename(columns={"team": "torvik_name"})
display(df_crosswalk.head(20))

,espn_id,espn_name,normalized_name,torvik_name,conf
0,222,Villanova,villanova,Villanova,BE
1,87,Notre Dame,notre dame,Notre Dame,ACC
2,57,Florida,florida,Florida,SEC
3,2390,Miami,miami,NaN,NaN
4,150,Duke,duke,Duke,ACC
5,349,Army,army,Army,Pat
6,97,Louisville,louisville,Louisville,ACC
7,236,Chattanooga,chattanooga,Chattanooga,SC
8,228,Clemson,clemson,Clemson,ACC
9,56,Stetson,stetson,Stetson,ASun


In [18]:
#identify unmatched teams
unmatched_teams = df_crosswalk[
    df_crosswalk["torvik_name"].isna()
]

print("Total ESPN teams:", len(df_espn_teams))

print("Successfully matched teams:",
      df_crosswalk["torvik_name"].notna().sum())

print("Unmatched teams:", len(unmatched_teams))

display(unmatched_teams[["espn_id", "espn_name"]])

Total ESPN teams: 116
Successfully matched teams: 110
Unmatched teams: 6


,espn_id,espn_name
3,2390,Miami
21,2001,Adams State
49,44,American University
74,357,Texas A&M-Corpus Christi
102,2352,Loyola Maryland
114,145,Ole Miss


In [19]:
#finding the correct Torvik names for unmatched teams
display(
    df_torvik[
        df_torvik["team"].str.contains(
            "Miami|Adams|American|Corpus|Loyola|Mississippi|Ole Miss",
            case=False,
            na=False
        )
    ][["team", "conf"]]
)

,team,conf
14,Miami FL,ACC
40,Mississippi,SEC
57,Mississippi St.,SEC
90,Miami OH,MAC
127,Loyola Chicago,A10
133,Loyola Marymount,WCC
159,American,Pat
218,Texas A&M Corpus Chris,Slnd
358,Loyola MD,Pat
361,Mississippi Valley St.,SWAC


In [20]:
#manually correct unmatched team names
manual_matches = {
    "Miami": "Miami FL",
    "American University": "American",
    "Texas A&M-Corpus Christi": "Texas A&M Corpus Chris",
    "Loyola Maryland": "Loyola MD",
    "Ole Miss": "Mississippi"
}

# Apply manual corrections
for espn_name, torvik_name in manual_matches.items():
    df_crosswalk.loc[
        df_crosswalk["espn_name"] == espn_name,
        "torvik_name"
    ] = torvik_name

# Update conference information
conference_map = df_torvik.set_index("team")["conf"]

df_crosswalk["conf"] = df_crosswalk["torvik_name"].map(conference_map)

# Check results after manual corrections
print("Total ESPN teams:", len(df_crosswalk))
print("Matched teams:", df_crosswalk["torvik_name"].notna().sum())
print("Unmatched teams:", df_crosswalk["torvik_name"].isna().sum())

display(df_crosswalk[df_crosswalk["torvik_name"].isna()])

Total ESPN teams: 116
Matched teams: 115
Unmatched teams: 1


,espn_id,espn_name,normalized_name,torvik_name,conf
21,2001,Adams State,adams state,NaN,NaN


In [21]:
#validate the team name crosswalk
#check that all 18 ACC schools are matched
expected_acc = set(df_torvik.loc[df_torvik["conf"] == "ACC", "team"])

matched_acc = set(df_crosswalk.loc[df_crosswalk["conf"] == "ACC", "torvik_name"])

assert len(expected_acc) == 18
assert matched_acc == expected_acc, "Missing ACC teams"

print("All 18 ACC teams are matched successfully.")

#check that no two ESPN teams map to the same Torvik team
matched_teams = df_crosswalk.dropna(subset=["torvik_name"])

assert matched_teams["torvik_name"].is_unique, "Duplicate Torvik mappings found"

print("No duplicate Torvik mappings found.")

#display all matched ACC teams
display(df_crosswalk[df_crosswalk["conf"] == "ACC"])

All 18 ACC teams are matched successfully.
No duplicate Torvik mappings found.


,espn_id,espn_name,normalized_name,torvik_name,conf
1,87,Notre Dame,notre dame,Notre Dame,ACC
3,2390,Miami,miami,Miami FL,ACC
4,150,Duke,duke,Duke,ACC
6,97,Louisville,louisville,Louisville,ACC
8,228,Clemson,clemson,Clemson,ACC
10,153,North Carolina,north carolina,North Carolina,ACC
12,183,Syracuse,syracuse,Syracuse,ACC
14,221,Pittsburgh,pittsburgh,Pittsburgh,ACC
16,152,NC State,nc state,N.C. State,ACC
18,2567,SMU,smu,SMU,ACC


 Decision Log: Team Name Matching

- Standardized team names by converting them to lowercase,
  removing punctuation, and treating "St." and "State"
  as equivalent. This automatically matched 110 of the
  116 ESPN teams.

- Manually corrected five unmatched team names:
  - Miami → Miami FL
  - American University → American
  - Texas A&M-Corpus Christi → Texas A&M Corpus Chris
  - Loyola Maryland → Loyola MD
  - Ole Miss → Mississippi

- Adams State was not found in the Torvik dataset.
  We decided to retain its game results but leave its
  rating missing. It will be excluded from average
  opponent rating calculations and counted separately
  as an unrated opponent. This affects one team.

- Validated that all 18 ACC teams were matched and that
  no two ESPN teams mapped to the same Torvik team.

- Final result: 115 matched teams and 1 unrated team.

**Game Filtering**

In [22]:
#keep only completed games
completed_games = []

for game in all_games:
    competition = game["competitions"][0]

    if competition["status"]["type"]["completed"]:
        completed_games.append(game)

print("Total games:", len(all_games))
print("Completed games:", len(completed_games))
print("Games removed:", len(all_games) - len(completed_games))

Total games: 110
Completed games: 110
Games removed: 0


In [23]:
#convert game start times from UTC to Eastern Time
game_dates = []

for game in completed_games:
    competition = game["competitions"][0]

    utc_time = pd.to_datetime(competition["date"], utc=True)

    eastern_time = utc_time.tz_convert("America/New_York")

    game_dates.append({
        "game_id": game["id"],
        "start_time_utc": utc_time,
        "start_time_et": eastern_time,
        "game_date": eastern_time.date()
    })

df_game_dates = pd.DataFrame(game_dates)

display(df_game_dates.head(10))

# Count games whose calendar dates changed
changed_dates = (
    df_game_dates["start_time_utc"].dt.date
    != df_game_dates["game_date"]
).sum()

print("Games with changed dates:", changed_dates)

,game_id,start_time_utc,start_time_et,game_date
0,401920982,2026-11-01 14:30:00+00:00,2026-11-01 09:30:00-05:00,2026-11-01
1,401906769,2026-11-02 23:00:00+00:00,2026-11-02 18:00:00-05:00,2026-11-02
2,401909562,2026-11-02 23:30:00+00:00,2026-11-02 18:30:00-05:00,2026-11-02
3,401918947,2026-11-03 02:00:00+00:00,2026-11-02 21:00:00-05:00,2026-11-02
4,401918073,2026-11-03 00:00:00+00:00,2026-11-02 19:00:00-05:00,2026-11-02
5,401913099,2026-11-03 00:00:00+00:00,2026-11-02 19:00:00-05:00,2026-11-02
6,401909716,2026-11-03 00:00:00+00:00,2026-11-02 19:00:00-05:00,2026-11-02
7,401909705,2026-11-03 00:00:00+00:00,2026-11-02 19:00:00-05:00,2026-11-02
8,401918955,2026-11-03 00:30:00+00:00,2026-11-02 19:30:00-05:00,2026-11-02
9,401918967,2026-11-03 01:00:00+00:00,2026-11-02 20:00:00-05:00,2026-11-02


Games with changed dates: 79


**Game Data Transformation**



In [24]:
#create one row per completed game
game_rows = []

for game in completed_games:
    competition = game["competitions"][0]

    # Identify home and away teams
    for competitor in competition["competitors"]:
        if competitor["homeAway"] == "home":
            home = competitor
        else:
            away = competitor

    game_rows.append({
        "game_id": game["id"],
        "home_team_id": home["team"]["id"],
        "away_team_id": away["team"]["id"],
        "home_score": int(home["score"]),
        "away_score": int(away["score"]),
        "neutral_site": competition["neutralSite"]
    })

df_games = pd.DataFrame(game_rows)

# Add our previously converted game dates
df_games = df_games.merge(
    df_game_dates,
    on="game_id",
    validate="one_to_one"
)

# Validate unique game IDs
assert df_games["game_id"].is_unique

print("Total game rows:", len(df_games))

display(df_games.head(10))

Total game rows: 110


,game_id,home_team_id,away_team_id,home_score,away_score,neutral_site,start_time_utc,start_time_et,game_date
0,401920982,222,87,72,74,True,2026-11-01 14:30:00+00:00,2026-11-01 09:30:00-05:00,2026-11-01
1,401906769,57,2390,77,67,True,2026-11-02 23:00:00+00:00,2026-11-02 18:00:00-05:00,2026-11-02
2,401909562,150,349,84,50,False,2026-11-02 23:30:00+00:00,2026-11-02 18:30:00-05:00,2026-11-02
3,401918947,97,236,84,70,False,2026-11-03 02:00:00+00:00,2026-11-02 21:00:00-05:00,2026-11-02
4,401918073,228,56,85,72,False,2026-11-03 00:00:00+00:00,2026-11-02 19:00:00-05:00,2026-11-02
5,401913099,153,2717,80,72,False,2026-11-03 00:00:00+00:00,2026-11-02 19:00:00-05:00,2026-11-02
6,401909716,183,2441,74,58,False,2026-11-03 00:00:00+00:00,2026-11-02 19:00:00-05:00,2026-11-02
7,401909705,221,2520,95,57,False,2026-11-03 00:00:00+00:00,2026-11-02 19:00:00-05:00,2026-11-02
8,401918955,152,2097,83,64,False,2026-11-03 00:30:00+00:00,2026-11-02 19:30:00-05:00,2026-11-02
9,401918967,2567,2582,85,72,False,2026-11-03 01:00:00+00:00,2026-11-02 20:00:00-05:00,2026-11-02


**Team-Level Game Transformation**

In [25]:
#create one row per team per game
team_game_rows = []

for index, game in df_games.iterrows():

    #home team perspective
    team_game_rows.append({
        "game_id": game["game_id"],
        "team_id": game["home_team_id"],
        "opponent_id": game["away_team_id"],
        "points_for": game["home_score"],
        "points_against": game["away_score"],
        "margin": game["home_score"] - game["away_score"],
        "location": "neutral" if game["neutral_site"] else "home"
    })

    #away team perspective
    team_game_rows.append({
        "game_id": game["game_id"],
        "team_id": game["away_team_id"],
        "opponent_id": game["home_team_id"],
        "points_for": game["away_score"],
        "points_against": game["home_score"],
        "margin": game["away_score"] - game["home_score"],
        "location": "neutral" if game["neutral_site"] else "away"
    })

df_team_games = pd.DataFrame(team_game_rows)

#validate that every game has two rows
assert len(df_team_games) == 2 * len(df_games)

assert (df_team_games.groupby("game_id").size() == 2).all()

print("Total games:", len(df_games))
print("Total team-game rows:", len(df_team_games))

display(df_team_games.head(10))

Total games: 110
Total team-game rows: 220


,game_id,team_id,opponent_id,points_for,points_against,margin,location
0,401920982,222,87,72,74,-2,neutral
1,401920982,87,222,74,72,2,neutral
2,401906769,57,2390,77,67,10,neutral
3,401906769,2390,57,67,77,-10,neutral
4,401909562,150,349,84,50,34,home
5,401909562,349,150,50,84,-34,away
6,401918947,97,236,84,70,14,home
7,401918947,236,97,70,84,-14,away
8,401918073,228,56,85,72,13,home
9,401918073,56,228,72,85,-13,away


**Team Box Score Statistics**

In [26]:
#split made-and-attempted statistics
def split_stat(value):
    made, attempted = value.split("-")
    return int(made), int(attempted)

#testing the function
fgm, fga = split_stat("29-63")

print("Field goals made:", fgm)
print("Field goals attempted:", fga)

Field goals made: 29
Field goals attempted: 63


In [27]:
#inspect available team statistics

for stat in sample_boxscore["boxscore"]["teams"][0]["statistics"]:
    print(stat["name"], ":", stat["displayValue"])

fieldGoalsMade-fieldGoalsAttempted : 26-56
fieldGoalPct : 46
threePointFieldGoalsMade-threePointFieldGoalsAttempted : 8-28
threePointFieldGoalPct : 29
freeThrowsMade-freeThrowsAttempted : 16-20
freeThrowPct : 80
totalRebounds : 37
offensiveRebounds : 9
defensiveRebounds : 28
assists : 16
steals : 10
blocks : 6
turnovers : 14
teamTurnovers : 0
totalTurnovers : 14
technicalFouls : 0
totalTechnicalFouls : 0
flagrantFouls : 0
turnoverPoints : 17
fastBreakPoints : 9
pointsInPaint : 34
fouls : 16
largestLead : 12
leadChanges : 1
leadPercentage : 22


In [28]:
#extract team statistics from saved box scores
team_stats_rows = []

for game_id in df_games["game_id"]:

    #open the saved JSON file
    file_path = boxscore_folder / f"{game_id}.json"

    with open(file_path, "r", encoding="utf-8") as file:
        game_data = json.load(file)

    for team in game_data["boxscore"]["teams"]:

        #convert statistics into a dictionary
        stats = {
            stat["name"]: stat["displayValue"]
            for stat in team["statistics"]
        }

        #separate made and attempted shots
        fgm, fga = split_stat(
            stats["fieldGoalsMade-fieldGoalsAttempted"]
        )

        tpm, tpa = split_stat(
            stats["threePointFieldGoalsMade-threePointFieldGoalsAttempted"]
        )

        ftm, fta = split_stat(
            stats["freeThrowsMade-freeThrowsAttempted"]
        )

        team_stats_rows.append({
            "game_id": game_id,
            "team_id": team["team"]["id"],
            "fgm": fgm,
            "fga": fga,
            "tpm": tpm,
            "tpa": tpa,
            "ftm": ftm,
            "fta": fta,
            "rebounds": int(stats["totalRebounds"]),
            "oreb": int(stats["offensiveRebounds"]),
            "dreb": int(stats["defensiveRebounds"]),
            "assists": int(stats["assists"]),
            "steals": int(stats["steals"]),
            "blocks": int(stats["blocks"]),
            "turnovers": int(stats["totalTurnovers"])
        })

df_team_stats = pd.DataFrame(team_stats_rows)
print("Total team statistics rows:", len(df_team_stats))
display(df_team_stats.head(10))

Total team statistics rows: 220


,game_id,team_id,fgm,fga,tpm,tpa,ftm,fta,rebounds,oreb,dreb,assists,steals,blocks,turnovers
0,401920982,87,23,65,9,24,19,25,37,17,20,14,14,2,11
1,401920982,222,24,52,11,31,13,18,37,10,27,12,6,4,20
2,401906769,2390,21,60,4,17,21,30,39,13,26,6,5,0,11
3,401906769,57,23,53,11,26,20,35,44,12,32,17,7,5,12
4,401909562,349,18,52,8,30,6,9,28,10,18,13,5,3,24
5,401909562,150,29,59,4,22,22,26,37,14,23,14,16,6,13
6,401918947,236,25,57,12,33,8,14,38,11,27,18,8,1,18
7,401918947,97,30,68,9,35,15,26,42,18,24,17,12,2,10
8,401918073,56,29,60,6,24,8,9,29,7,22,13,6,4,9
9,401918073,228,25,49,9,24,26,35,32,8,24,17,5,5,8


**Estimated Possessions**

In [29]:
#calculate estimated possessions for each team

df_team_stats["estimated_possessions"] = (
    df_team_stats["fga"]
    - df_team_stats["oreb"]
    + df_team_stats["turnovers"]
    + 0.475 * df_team_stats["fta"]
)

#average the two teams' estimates for each game
df_team_stats["game_possessions"] = (
    df_team_stats.groupby("game_id")["estimated_possessions"]
    .transform("mean")
)

#validate that possessions are positive
assert (df_team_stats["estimated_possessions"] > 0).all()

display(df_team_stats[
    ["game_id", "team_id", "fga", "oreb", "turnovers",
     "fta", "estimated_possessions", "game_possessions"]
].head(10))

,game_id,team_id,fga,oreb,turnovers,fta,estimated_possessions,game_possessions
0,401920982,87,65,17,11,25,70.875,70.7125
1,401920982,222,52,10,20,18,70.550,70.7125
2,401906769,2390,60,13,11,30,72.250,70.9375
3,401906769,57,53,12,12,35,69.625,70.9375
4,401909562,349,52,10,24,9,70.275,70.3125
5,401909562,150,59,14,13,26,70.350,70.3125
6,401918947,236,57,11,18,14,70.650,71.5000
7,401918947,97,68,18,10,26,72.350,71.5000
8,401918073,56,60,7,9,9,66.275,65.9500
9,401918073,228,49,8,8,35,65.625,65.9500


In [30]:
#combine team game information and box score statistics
df_team_games_final = df_team_games.merge(
    df_team_stats,
    on=["game_id", "team_id"],
    how="left",
    validate="one_to_one"
)

#validate the merged DataFrame
assert len(df_team_games_final) == 2 * len(df_games)

assert df_team_games_final["game_possessions"].notna().all()

print("Total rows:", len(df_team_games_final))
print("Total columns:", len(df_team_games_final.columns))
print("Missing possession values:",
      df_team_games_final["game_possessions"].isna().sum())

display(df_team_games_final.head(10))

Total rows: 220
Total columns: 22
Missing possession values: 0


,game_id,team_id,opponent_id,points_for,points_against,margin,location,fgm,fga,tpm,...,fta,rebounds,oreb,dreb,assists,steals,blocks,turnovers,estimated_possessions,game_possessions
0,401920982,222,87,72,74,-2,neutral,24,52,11,...,18,37,10,27,12,6,4,20,70.550,70.7125
1,401920982,87,222,74,72,2,neutral,23,65,9,...,25,37,17,20,14,14,2,11,70.875,70.7125
2,401906769,57,2390,77,67,10,neutral,23,53,11,...,35,44,12,32,17,7,5,12,69.625,70.9375
3,401906769,2390,57,67,77,-10,neutral,21,60,4,...,30,39,13,26,6,5,0,11,72.250,70.9375
4,401909562,150,349,84,50,34,home,29,59,4,...,26,37,14,23,14,16,6,13,70.350,70.3125
5,401909562,349,150,50,84,-34,away,18,52,8,...,9,28,10,18,13,5,3,24,70.275,70.3125
6,401918947,97,236,84,70,14,home,30,68,9,...,26,42,18,24,17,12,2,10,72.350,71.5000
7,401918947,236,97,70,84,-14,away,25,57,12,...,14,38,11,27,18,8,1,18,70.650,71.5000
8,401918073,228,56,85,72,13,home,25,49,9,...,35,32,8,24,17,5,5,8,65.625,65.9500
9,401918073,56,228,72,85,-13,away,29,60,6,...,9,29,7,22,13,6,4,9,66.275,65.9500


**Player Statistics Transformation**


In [31]:
#inspect player statistics structure
sample_players = sample_boxscore["boxscore"]["players"][0]

player_stats = sample_players["statistics"][0]

print("Statistic keys:")
print(player_stats["keys"])

print("\nSample player:")
print(player_stats["athletes"][0])

Statistic keys:
['minutes', 'points', 'fieldGoalsMade-fieldGoalsAttempted', 'threePointFieldGoalsMade-threePointFieldGoalsAttempted', 'freeThrowsMade-freeThrowsAttempted', 'rebounds', 'assists', 'turnovers', 'steals', 'blocks', 'offensiveRebounds', 'defensiveRebounds', 'fouls']

Sample player:
{'active': False, 'athlete': {'id': '5174559', 'uid': 's:40~l:41~a:5174559', 'guid': '326ae117-60ab-3a69-a01e-fcdfde8bf4ff', 'displayName': 'Akir Souare', 'shortName': 'A. Souare', 'links': [{'rel': ['playercard', 'desktop', 'athlete'], 'href': 'https://www.espn.com/mens-college-basketball/player/_/id/5174559/akir-souare', 'text': 'Player Card'}], 'headshot': {'href': 'https://a.espncdn.com/i/headshots/mens-college-basketball/players/full/5174559.png', 'alt': 'Akir Souare'}, 'jersey': '4', 'position': {'name': 'Forward', 'displayName': 'Forward', 'abbreviation': 'F'}}, 'starter': True, 'didNotPlay': False, 'ejected': False, 'stats': ['12', '4', '1-2', '0-0', '2-2', '3', '0', '0', '2', '1', '3', '

In [32]:
#extract individual player statistics
player_rows = []
skipped_players = 0

for game_id in df_games["game_id"]:

    file_path = boxscore_folder / f"{game_id}.json"

    with open(file_path, "r", encoding="utf-8") as file:
        game_data = json.load(file)

    for team in game_data["boxscore"]["players"]:

        team_id = team["team"]["id"]

        for group in team["statistics"]:

            keys = group["keys"]

            for player in group["athletes"]:

                # Skip players who did not participate
                if player["didNotPlay"]:
                    skipped_players += 1
                    continue

                # Match statistic names with their values
                assert len(keys) == len(player["stats"])

                stats = dict(zip(keys, player["stats"]))

                player_rows.append({
                    "game_id": game_id,
                    "team_id": team_id,
                    "player_id": player["athlete"]["id"],
                    "player_name": player["athlete"]["displayName"],
                    "minutes": int(stats["minutes"]),
                    "points": int(stats["points"]),
                    "rebounds": int(stats["rebounds"]),
                    "assists": int(stats["assists"]),
                    "steals": int(stats["steals"]),
                    "blocks": int(stats["blocks"]),
                    "turnovers": int(stats["turnovers"])
                })

df_player_games = pd.DataFrame(player_rows)

print("Total player-game rows:", len(df_player_games))
print("Total columns:", len(df_player_games.columns))
print("Players who did not play:", skipped_players)

display(df_player_games.head(10))

Total player-game rows: 2487
Total columns: 11
Players who did not play: 917


,game_id,team_id,player_id,player_name,minutes,points,rebounds,assists,steals,blocks,turnovers
0,401920982,87,5312256,Brady Koehler,25,16,8,1,2,2,0
1,401920982,87,4682856,Logan Duncomb,21,9,3,4,1,0,3
2,401920982,87,5108811,Ethan Roberts,32,11,2,0,1,0,1
3,401920982,87,5105672,Braeden Smith,31,16,4,3,4,0,1
4,401920982,87,4912455,Braeden Shrewsberry,36,15,6,3,1,0,0
5,401920982,87,5144387,Bryce Ruglass Dortch,5,0,2,0,0,0,0
6,401920982,87,5107470,Yoro Diallo,8,0,2,0,0,0,0
7,401920982,87,5312259,Tommy Ahneman,13,4,6,0,1,0,1
8,401920982,87,5311849,Devin Brown,4,0,0,0,0,0,1
9,401920982,87,5210410,Jonathan Sanderson,10,3,1,2,1,0,2


In [33]:
#validate individual player statistics
duplicates = df_player_games.duplicated(
    subset=["game_id", "player_id"]
).sum()

missing_values = df_player_games.isna().sum().sum()

stat_columns = [
    "minutes", "points", "rebounds",
    "assists", "steals", "blocks", "turnovers"
]

assert duplicates == 0
assert missing_values == 0
assert (df_player_games[stat_columns] >= 0).all().all()

assert set(df_player_games["game_id"]) == set(df_games["game_id"])

#verify that both teams in every game have player records
player_team_games = set(zip(
    df_player_games["game_id"],
    df_player_games["team_id"]
))

expected_team_games = set(zip(
    df_team_games["game_id"],
    df_team_games["team_id"]
))

assert player_team_games == expected_team_games

print("Duplicate player-game rows:", duplicates)
print("Missing values:", missing_values)
print("Games represented:", df_player_games["game_id"].nunique())
print("Team-game combinations:", len(player_team_games))
print("All player statistics passed validation.")

Duplicate player-game rows: 0
Missing values: 0
Games represented: 110
Team-game combinations: 220
All player statistics passed validation.


In [34]:
#count rows affected by transformation decisions
unrated_ids = df_crosswalk.loc[
    df_crosswalk["torvik_name"].isna(), "espn_id"
]

unrated_opponent_rows = df_team_games[
    df_team_games["opponent_id"].isin(unrated_ids)
]

print("Games removed:", len(all_games) - len(completed_games))
print("Games with corrected dates:", changed_dates)
print("Team-game rows created:", len(df_team_games_final))
print("Neutral-site games:", df_games["neutral_site"].sum())
print("Neutral-site team rows:",
      (df_team_games["location"] == "neutral").sum())
print("Rows with unrated opponents:", len(unrated_opponent_rows))
print("Player-game rows created:", len(df_player_games))
print("Did-not-play records excluded:", skipped_players)

Games removed: 0
Games with corrected dates: 79
Team-game rows created: 220
Neutral-site games: 19
Neutral-site team rows: 38
Rows with unrated opponents: 1
Player-game rows created: 2487
Did-not-play records excluded: 917


**Decision Log: Game and Player Transformation**

- **Game filtering:** Only completed games were retained.
  All 110 games were already completed, so 0 games
  were removed during this filtering step.

- **Time zone conversion:** Converted UTC start times
  to Eastern Time before extracting the official game
  date. This corrected the dates of 79 games.

- **Game restructuring:** Converted 110 game records
  into 220 team-game records, representing each game
  from both teams' perspectives.

- **Neutral-site games:** Identified 19 neutral-site
  games and labeled both teams' locations as neutral.
  This affected 38 team-game rows.

- **Unrated opponents:** Retained games involving
  Adams State but left its rating missing. This
  affected 1 team-game row with an unrated opponent.

- **Shooting statistics:** Separated made-and-attempted
  statistics into individual numeric columns for
  all 220 team-game rows.

- **Estimated possessions:** Used the formula provided
  in Appendix A and averaged both teams' estimates
  to obtain a shared possession estimate for each game.
  This affected 220 team-game rows.

- **Player statistics:** Created 2,487 player-game
  records and excluded 917 did-not-play records
  to avoid distorting per-game averages.

- **Data validation:** Verified that there were no
  duplicate player-game records, no missing player
  statistics, and that all 110 games and 220 team-game
  combinations were represented.

**Load SQLite Database**

In [36]:
#connect to the SQLite database
import sqlite3
conn = sqlite3.connect("acc.db")
cursor = conn.cursor()
#enable foreign key constraints. creates a db if it doesnt already exist
cursor.execute("PRAGMA foreign_keys = ON")
print("Successfully connected to acc.db")

Successfully connected to acc.db


**Preparing Teams Table**
Combine all Division I teams from the Torvik dataset
with the teams collected from ESPN. Include unique
team identifiers, ESPN names, Torvik names, conference
information, and an ACC membership flag.

In [37]:
#repare the teams DataFrame
df_teams = df_torvik[["team", "conf"]].copy()
df_teams = df_teams.rename(columns={"team": "torvik_name"})

#combine Torvik teams with matched ESPN teams
df_teams = df_teams.merge(
    df_crosswalk[["espn_id", "espn_name", "torvik_name"]],
    on="torvik_name",
    how="outer",
    validate="one_to_one"
)

#identify ACC teams
df_teams["is_acc"] = (df_teams["conf"] == "ACC").astype(int)

#create stable internal team IDs
df_teams["team_id"] = df_teams.apply(
    lambda row: "torvik:" + row["torvik_name"]
    if pd.notna(row["torvik_name"])
    else "espn:" + str(row["espn_id"]),
    axis=1
)
#organize columns
df_teams = df_teams[
    ["team_id", "espn_id", "espn_name",
     "torvik_name", "conf", "is_acc"]
]

#validation
assert df_teams["team_id"].is_unique
assert df_teams["is_acc"].sum() == 18

print("Total teams:", len(df_teams))
print("ACC teams:", df_teams["is_acc"].sum())
print("Teams with ESPN IDs:", df_teams["espn_id"].notna().sum())
print("Teams with Torvik ratings:", df_teams["torvik_name"].notna().sum())
display(df_teams.head(10))

Total teams: 366
ACC teams: 18
Teams with ESPN IDs: 116
Teams with Torvik ratings: 365


,team_id,espn_id,espn_name,torvik_name,conf,is_acc
0,torvik:Abilene Christian,NaN,NaN,Abilene Christian,UAC,0
1,torvik:Air Force,NaN,NaN,Air Force,MWC,0
2,torvik:Akron,NaN,NaN,Akron,MAC,0
3,torvik:Alabama,NaN,NaN,Alabama,SEC,0
4,torvik:Alabama A&M,NaN,NaN,Alabama A&M,SWAC,0
5,torvik:Alabama St.,2011,Alabama State,Alabama St.,SWAC,0
6,torvik:Albany,NaN,NaN,Albany,AE,0
7,torvik:Alcorn St.,NaN,NaN,Alcorn St.,SWAC,0
8,torvik:American,44,American University,American,Pat,0
9,torvik:Appalachian St.,NaN,NaN,Appalachian St.,SB,0


**Prepare Team Crosswalk Table**

In [38]:
#Prepare the team crosswalk for SQLite
crosswalk_rows = []

for index, team in df_teams.iterrows():

    #add ESPN team name
    if pd.notna(team["espn_name"]):
        crosswalk_rows.append({
            "source": "ESPN",
            "source_name": team["espn_name"],
            "team_id": team["team_id"]
        })

    #add Torvik team name
    if pd.notna(team["torvik_name"]):
        crosswalk_rows.append({
            "source": "Torvik",
            "source_name": team["torvik_name"],
            "team_id": team["team_id"]
        })

df_team_crosswalk = pd.DataFrame(crosswalk_rows)

#validate unique source-name combinations
assert not df_team_crosswalk.duplicated(
    subset=["source", "source_name"]
).any()

print("Total crosswalk rows:", len(df_team_crosswalk))
print(df_team_crosswalk["source"].value_counts())
display(df_team_crosswalk.head(15))

Total crosswalk rows: 481
source
Torvik    365
ESPN      116
Name: count, dtype: int64


,source,source_name,team_id
0,Torvik,Abilene Christian,torvik:Abilene Christian
1,Torvik,Air Force,torvik:Air Force
2,Torvik,Akron,torvik:Akron
3,Torvik,Alabama,torvik:Alabama
4,Torvik,Alabama A&M,torvik:Alabama A&M
5,ESPN,Alabama State,torvik:Alabama St.
6,Torvik,Alabama St.,torvik:Alabama St.
7,Torvik,Albany,torvik:Albany
8,Torvik,Alcorn St.,torvik:Alcorn St.
9,ESPN,American University,torvik:American


**Prepare Ratings Table**

In [39]:
#identify available Torvik rating snapshots
from pathlib import Path
ratings_folder = Path("data/raw/torvik")
snapshot_files = sorted(
    ratings_folder.glob("torvik_*.csv")
)
print("Number of saved snapshots:", len(snapshot_files))

for file in snapshot_files:
    print(file.name)

Number of saved snapshots: 1
torvik_2026-11-28.csv


In [40]:
#prepare ratings from all saved snapshots
ratings_frames = []
for file in snapshot_files:
    #read the saved CSV
    snapshot = pd.read_csv(file)

    #extract the date from the filename
    snapshot_date = file.stem.replace("torvik_", "")

    #select the required columns
    ratings = snapshot[
        ["team", "record", "adjoe", "adjde",
         "barthag", "adjt", "sos", "rank.1"]
    ].copy()

    ratings = ratings.rename(columns={
        "team": "torvik_name",
        "rank.1": "rank"
    })

    #connect ratings to internal team IDs
    ratings = ratings.merge(
        df_teams[["team_id", "torvik_name"]].dropna(
            subset=["torvik_name"]
        ),
        on="torvik_name",
        how="left",
        validate="one_to_one"
    )

    ratings["snapshot_date"] = snapshot_date

    #calculate efficiency margin
    ratings["efficiency_margin"] = (
        ratings["adjoe"] - ratings["adjde"]
    )

    ratings_frames.append(ratings)

df_ratings = pd.concat(ratings_frames, ignore_index=True)

#validate ratings
assert df_ratings["team_id"].notna().all()

assert not df_ratings.duplicated(
    subset=["team_id", "snapshot_date"]
).any()

print("Total ratings rows:", len(df_ratings))
print("Unique rated teams:", df_ratings["team_id"].nunique())
print("Number of snapshots:", df_ratings["snapshot_date"].nunique())

display(df_ratings.head(10))

Total ratings rows: 365
Unique rated teams: 365
Number of snapshots: 1


,torvik_name,record,adjoe,adjde,barthag,adjt,sos,rank,team_id,snapshot_date,efficiency_margin
0,Houston,6-0,124.474683,93.804798,0.962788,67.558509,0.671798,1,torvik:Houston,2026-11-28,30.669885
1,Illinois,5-1,126.281293,96.602043,0.956102,73.530920,0.514516,2,torvik:Illinois,2026-11-28,29.679250
2,Kansas,5-1,122.691516,94.294347,0.953797,72.710877,0.692280,3,torvik:Kansas,2026-11-28,28.397169
3,Florida,6-1,124.762015,96.285008,0.951644,68.495192,0.710338,4,torvik:Florida,2026-11-28,28.477007
4,Duke,4-2,119.887207,93.240421,0.947388,71.564028,0.490622,5,torvik:Duke,2026-11-28,26.646786
5,Virginia,7-0,123.771587,96.753644,0.944386,70.050165,0.413679,6,torvik:Virginia,2026-11-28,27.017943
6,Arizona,4-1,121.325480,94.865025,0.944236,71.810337,0.647468,7,torvik:Arizona,2026-11-28,26.460455
7,Connecticut,4-1,120.006300,94.540143,0.939515,66.970154,0.580697,8,torvik:Connecticut,2026-11-28,25.466156
8,Texas Tech,5-0,120.660428,95.675178,0.935127,68.230885,0.358033,9,torvik:Texas Tech,2026-11-28,24.985250
9,Michigan St.,5-1,119.780692,95.050444,0.934590,68.469491,0.506158,10,torvik:Michigan St.,2026-11-28,24.730248


**Map Internal Team Identifiers**

In [41]:
#create a mapping from ESPN IDs to internal team IDs
espn_to_internal = (
    df_teams.dropna(subset=["espn_id"])
    .set_index("espn_id")["team_id"]
    .to_dict()
)

#create copies to preserve the original DataFrames
df_games_db = df_games.copy()
df_team_games_db = df_team_games_final.copy()
df_player_games_db = df_player_games.copy()

#map team IDs in games
df_games_db["home_team_id"] = (
    df_games_db["home_team_id"].map(espn_to_internal)
)

df_games_db["away_team_id"] = (
    df_games_db["away_team_id"].map(espn_to_internal)
)

#map team and opponent IDs in team_games
df_team_games_db["team_id"] = (
    df_team_games_db["team_id"].map(espn_to_internal)
)

df_team_games_db["opponent_id"] = (
    df_team_games_db["opponent_id"].map(espn_to_internal)
)

#map team IDs in player_games
df_player_games_db["team_id"] = (
    df_player_games_db["team_id"].map(espn_to_internal)
)

#validate that all IDs were successfully mapped
assert df_games_db["home_team_id"].notna().all()
assert df_games_db["away_team_id"].notna().all()
assert df_team_games_db["team_id"].notna().all()
assert df_team_games_db["opponent_id"].notna().all()
assert df_player_games_db["team_id"].notna().all()

print("All team IDs successfully mapped.")
print("Game rows:", len(df_games_db))
print("Team-game rows:", len(df_team_games_db))
print("Player-game rows:", len(df_player_games_db))

display(df_games_db.head())

All team IDs successfully mapped.
Game rows: 110
Team-game rows: 220
Player-game rows: 2487


,game_id,home_team_id,away_team_id,home_score,away_score,neutral_site,start_time_utc,start_time_et,game_date
0,401920982,torvik:Villanova,torvik:Notre Dame,72,74,True,2026-11-01 14:30:00+00:00,2026-11-01 09:30:00-05:00,2026-11-01
1,401906769,torvik:Florida,torvik:Miami FL,77,67,True,2026-11-02 23:00:00+00:00,2026-11-02 18:00:00-05:00,2026-11-02
2,401909562,torvik:Duke,torvik:Army,84,50,False,2026-11-02 23:30:00+00:00,2026-11-02 18:30:00-05:00,2026-11-02
3,401918947,torvik:Louisville,torvik:Chattanooga,84,70,False,2026-11-03 02:00:00+00:00,2026-11-02 21:00:00-05:00,2026-11-02
4,401918073,torvik:Clemson,torvik:Stetson,85,72,False,2026-11-03 00:00:00+00:00,2026-11-02 19:00:00-05:00,2026-11-02


**Create SQLite Tables**

Create the six required database tables using SQL.
Define primary keys to prevent duplicate records and
foreign keys to maintain relationships between tables.

In [42]:
#create SQLite database tables
cursor.executescript("""

CREATE TABLE IF NOT EXISTS teams (
    team_id TEXT PRIMARY KEY NOT NULL,
    espn_id TEXT UNIQUE,
    espn_name TEXT,
    torvik_name TEXT UNIQUE,
    conf TEXT,
    is_acc INTEGER NOT NULL
);

CREATE TABLE IF NOT EXISTS team_crosswalk (
    source TEXT NOT NULL,
    source_name TEXT NOT NULL,
    team_id TEXT NOT NULL,
    PRIMARY KEY (source, source_name),
    FOREIGN KEY (team_id) REFERENCES teams(team_id)
);

CREATE TABLE IF NOT EXISTS ratings (
    team_id TEXT NOT NULL,
    snapshot_date TEXT NOT NULL,
    torvik_name TEXT,
    record TEXT,
    adjoe REAL,
    adjde REAL,
    barthag REAL,
    adjt REAL,
    sos REAL,
    rank INTEGER,
    efficiency_margin REAL,
    PRIMARY KEY (team_id, snapshot_date),
    FOREIGN KEY (team_id) REFERENCES teams(team_id)
);

CREATE TABLE IF NOT EXISTS games (
    game_id TEXT PRIMARY KEY NOT NULL,
    home_team_id TEXT NOT NULL,
    away_team_id TEXT NOT NULL,
    home_score INTEGER,
    away_score INTEGER,
    neutral_site INTEGER,
    start_time_utc TEXT,
    start_time_et TEXT,
    game_date TEXT,
    FOREIGN KEY (home_team_id) REFERENCES teams(team_id),
    FOREIGN KEY (away_team_id) REFERENCES teams(team_id)
);

CREATE TABLE IF NOT EXISTS team_games (
    game_id TEXT NOT NULL,
    team_id TEXT NOT NULL,
    opponent_id TEXT NOT NULL,
    points_for INTEGER,
    points_against INTEGER,
    margin INTEGER,
    location TEXT,
    fgm INTEGER,
    fga INTEGER,
    tpm INTEGER,
    tpa INTEGER,
    ftm INTEGER,
    fta INTEGER,
    rebounds INTEGER,
    oreb INTEGER,
    dreb INTEGER,
    assists INTEGER,
    steals INTEGER,
    blocks INTEGER,
    turnovers INTEGER,
    estimated_possessions REAL,
    game_possessions REAL,
    PRIMARY KEY (game_id, team_id),
    FOREIGN KEY (game_id) REFERENCES games(game_id),
    FOREIGN KEY (team_id) REFERENCES teams(team_id),
    FOREIGN KEY (opponent_id) REFERENCES teams(team_id)
);

CREATE TABLE IF NOT EXISTS player_games (
    game_id TEXT NOT NULL,
    team_id TEXT NOT NULL,
    player_id TEXT NOT NULL,
    player_name TEXT,
    minutes INTEGER,
    points INTEGER,
    rebounds INTEGER,
    assists INTEGER,
    steals INTEGER,
    blocks INTEGER,
    turnovers INTEGER,
    PRIMARY KEY (game_id, player_id),
    FOREIGN KEY (game_id, team_id)
        REFERENCES team_games(game_id, team_id)
);

""")

conn.commit()

print("All six SQLite tables created successfully.")

All six SQLite tables created successfully.


In [44]:
#verify database tables
tables = pd.read_sql_query("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
""", conn)

print("Total database tables:", len(tables))
display(tables)

Total database tables: 6


,name
0,games
1,player_games
2,ratings
3,team_crosswalk
4,team_games
5,teams


**Load Data into SQLite**

In [46]:
#reusable function to load DataFrames into SQLite

def load_table(table_name, dataframe):

    #create a copy to preserve the original DataFrame
    data = dataframe.copy()

    #convert dates and timestamps to strings
    for column in data.columns:
        data[column] = data[column].map(
            lambda value: value.isoformat()
            if hasattr(value, "isoformat") else value
        )

    #convert missing values to None for SQLite
    data = data.astype(object).where(pd.notna(data), None)

    columns = ", ".join(data.columns)
    placeholders = ", ".join(["?"] * len(data.columns))

    query = f"""
        INSERT OR REPLACE INTO {table_name}
        ({columns})
        VALUES ({placeholders})
    """

    cursor.executemany(
        query,
        list(data.itertuples(index=False, name=None))
    )

    print(f"{table_name}: {len(data)} rows processed")

In [48]:
#load all six tables
def load_database():

    with conn:
        load_table("teams", df_teams)
        load_table("team_crosswalk", df_team_crosswalk)
        load_table("ratings", df_ratings)
        load_table("games", df_games_db)
        load_table("team_games", df_team_games_db)
        load_table("player_games", df_player_games_db)

    print("\nDatabase loaded successfully!")
load_database()

teams: 366 rows processed
team_crosswalk: 481 rows processed
ratings: 365 rows processed
games: 110 rows processed
team_games: 220 rows processed
player_games: 2487 rows processed

Database loaded successfully!


**Database Validation and Idempotency**

In [49]:
#verify actual database row counts
table_names = [
    "teams",
    "team_crosswalk",
    "ratings",
    "games",
    "team_games",
    "player_games"
]

def get_row_counts():

    counts = {}

    for table in table_names:
        cursor.execute(f"SELECT COUNT(*) FROM {table}")
        counts[table] = cursor.fetchone()[0]

    return counts

counts_before = get_row_counts()

display(pd.DataFrame(
    counts_before.items(),
    columns=["Table", "Row Count"]
))

,Table,Row Count
0,teams,366
1,team_crosswalk,481
2,ratings,365
3,games,110
4,team_games,220
5,player_games,2487


In [50]:
#run the loading process a second time
load_database()

#get row counts after the second load
counts_after = get_row_counts()

#compare both runs
df_load_test = pd.DataFrame({
    "Table": table_names,
    "First Load": [counts_before[t] for t in table_names],
    "Second Load": [counts_after[t] for t in table_names]
})

df_load_test["Unchanged"] = (
    df_load_test["First Load"] == df_load_test["Second Load"]
)

display(df_load_test)

#validate idempotency
assert counts_before == counts_after

print("Idempotency test passed: all row counts remained unchanged.")

teams: 366 rows processed
team_crosswalk: 481 rows processed
ratings: 365 rows processed
games: 110 rows processed
team_games: 220 rows processed
player_games: 2487 rows processed

Database loaded successfully!


,Table,First Load,Second Load,Unchanged
0,teams,366,366,True
1,team_crosswalk,481,481,True
2,ratings,365,365,True
3,games,110,110,True
4,team_games,220,220,True
5,player_games,2487,2487,True


Idempotency test passed: all row counts remained unchanged.


In [51]:
#final database integrity checks

#check foreign key relationships
cursor.execute("PRAGMA foreign_key_check")
foreign_key_errors = cursor.fetchall()

#check database integrity
cursor.execute("PRAGMA integrity_check")
integrity_result = cursor.fetchone()[0]

#verify ACC team count
cursor.execute("SELECT COUNT(*) FROM teams WHERE is_acc = 1")
acc_count = cursor.fetchone()[0]

#check that every game has two team-game rows
cursor.execute("""
    SELECT g.game_id
    FROM games g
    LEFT JOIN team_games tg
        ON g.game_id = tg.game_id
    GROUP BY g.game_id
    HAVING COUNT(tg.team_id) != 2
""")

invalid_games = cursor.fetchall()

#assertions
assert len(foreign_key_errors) == 0
assert integrity_result == "ok"
assert acc_count == 18
assert len(invalid_games) == 0

print("Foreign key errors:", len(foreign_key_errors))
print("Database integrity:", integrity_result)
print("ACC teams:", acc_count)
print("Invalid game records:", len(invalid_games))
print("All database validation checks passed!")

Foreign key errors: 0
Database integrity: ok
ACC teams: 18
Invalid game records: 0
All database validation checks passed!


**Database Validation Results**

Successfully created and populated all six required
SQLite tables in acc.db.

The loading process was executed twice using
INSERT OR REPLACE. All table row counts remained
unchanged, demonstrating idempotency.

The database passed all integrity checks, with zero
foreign key errors, all 18 ACC teams represented,
and exactly two team-game records per game.

The current database contains:
- 366 teams
- 481 team crosswalk records
- 365 rating records
- 110 completed games
- 220 team-game records
- 2,487 player-game records

**SQL Analysis**

 Q1. Record versus Rating

Compare each ACC team's winning percentage with its
adjusted efficiency margin using the newest available
Torvik ratings snapshot. Identify teams whose records
may overstate or understate their performance.

In [52]:
# Q1: Calculate ACC team records using SQL
query_q1_records = """
SELECT
    t.team_id,
    t.espn_name AS team,
    COUNT(*) AS games_played,
    SUM(CASE WHEN tg.margin > 0 THEN 1 ELSE 0 END) AS wins,
    SUM(CASE WHEN tg.margin < 0 THEN 1 ELSE 0 END) AS losses,
    ROUND(
        100.0 * SUM(CASE WHEN tg.margin > 0 THEN 1 ELSE 0 END)
        / COUNT(*), 2
    ) AS win_pct
FROM team_games tg
JOIN teams t
    ON tg.team_id = t.team_id
WHERE t.is_acc = 1
GROUP BY t.team_id, t.espn_name
ORDER BY win_pct DESC, team;
"""

df_q1_records = pd.read_sql_query(query_q1_records, conn)
assert len(df_q1_records) == 18
display(df_q1_records)

,team_id,team,games_played,wins,losses,win_pct
0,torvik:Boston College,Boston College,6,6,0,100.00
1,torvik:Pittsburgh,Pittsburgh,6,6,0,100.00
2,torvik:SMU,SMU,6,6,0,100.00
3,torvik:Virginia,Virginia,7,7,0,100.00
4,torvik:Notre Dame,Notre Dame,7,6,1,85.71
5,torvik:Virginia Tech,Virginia Tech,7,6,1,85.71
6,torvik:Florida St.,Florida State,6,5,1,83.33
7,torvik:Georgia Tech,Georgia Tech,6,5,1,83.33
8,torvik:Miami FL,Miami,6,5,1,83.33
9,torvik:N.C. State,NC State,6,5,1,83.33
